# Split MNIST into 100 Clients (60/class) + Disjoint 1k Test Set

This notebook creates:
- A **60,000-sample training pool** with balanced classes (6,000 per class)
- **100 clients** with 600 samples each (60 per class)
- A separate **1,000-sample test set** (100 per class)

The test set is selected from indices not used in the 60,000 training pool, so there is no overlap.

In [1]:
import os
import numpy as np
from tensorflow.keras.datasets import mnist
from sklearn.utils import shuffle

In [2]:
# Reproducibility
SEED = 42
np.random.seed(SEED)

# Configuration
NUM_CLASSES = 10
NUM_CLIENTS = 100
TRAIN_PER_CLASS = 6000          # 60,000 total
CLIENT_PER_CLASS = 60           # 600 samples/client
TEST_PER_CLASS = 100            # 1,000 total

OUTPUT_DIR = 'mnist_100_clients_60_per_class_1k_test'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Configuration loaded.')
print(f'Training pool: {TRAIN_PER_CLASS * NUM_CLASSES}')
print(f'Clients: {NUM_CLIENTS}, per client: {CLIENT_PER_CLASS * NUM_CLASSES}')
print(f'Test set: {TEST_PER_CLASS * NUM_CLASSES}')

Configuration loaded.
Training pool: 60000
Clients: 100, per client: 600
Test set: 1000


In [3]:
# Load and combine MNIST train/test to build a 70k unified source
(x_train, y_train), (x_test, y_test) = mnist.load_data()
all_x = np.concatenate([x_train, x_test], axis=0)
all_y = np.concatenate([y_train, y_test], axis=0)

print(f'Combined data shape: {all_x.shape}')
print(f'Combined labels shape: {all_y.shape}')

Combined data shape: (70000, 28, 28)
Combined labels shape: (70000,)


In [4]:
# Shuffle globally once, then select per class
all_x, all_y = shuffle(all_x, all_y, random_state=SEED)

# Keep track of source indices to guarantee disjoint train/test selections
all_indices = np.arange(len(all_y))

x_train_parts, y_train_parts = [], []
x_test_parts, y_test_parts = [], []
selected_train_indices = []
selected_test_indices = []

for class_label in range(NUM_CLASSES):
    class_idx = np.where(all_y == class_label)[0]
    if len(class_idx) < TRAIN_PER_CLASS + TEST_PER_CLASS:
        raise ValueError(f'Not enough samples for class {class_label}. Found {len(class_idx)}.')

    # First 6000/class -> training pool
    train_idx = class_idx[:TRAIN_PER_CLASS]
    # Next 100/class from remaining -> disjoint test set
    test_idx = class_idx[TRAIN_PER_CLASS:TRAIN_PER_CLASS + TEST_PER_CLASS]

    x_train_parts.append(all_x[train_idx])
    y_train_parts.append(all_y[train_idx])
    x_test_parts.append(all_x[test_idx])
    y_test_parts.append(all_y[test_idx])

    selected_train_indices.append(train_idx)
    selected_test_indices.append(test_idx)

x_selected = np.concatenate(x_train_parts, axis=0)
y_selected = np.concatenate(y_train_parts, axis=0)
x_test_1000 = np.concatenate(x_test_parts, axis=0)
y_test_1000 = np.concatenate(y_test_parts, axis=0)

selected_train_indices = np.concatenate(selected_train_indices)
selected_test_indices = np.concatenate(selected_test_indices)

print(f'Training pool shape: {x_selected.shape}, labels: {y_selected.shape}')
print(f'Test set shape: {x_test_1000.shape}, labels: {y_test_1000.shape}')

Training pool shape: (60000, 28, 28), labels: (60000,)
Test set shape: (1000, 28, 28), labels: (1000,)


In [5]:
# Verify disjointness by source indices
overlap = np.intersect1d(selected_train_indices, selected_test_indices)
print(f'Overlap count between 60k training pool and 1k test set: {len(overlap)}')
assert len(overlap) == 0, 'Train and test sets overlap, which should never happen.'
print('Disjointness check passed.')

Overlap count between 60k training pool and 1k test set: 0
Disjointness check passed.


In [6]:
# Save the complete selected datasets
train_file = os.path.join(OUTPUT_DIR, 'complete_60000_samples.npz')
test_file = os.path.join(OUTPUT_DIR, 'test_1000_samples.npz')

np.savez(train_file, x=x_selected, y=y_selected)
np.savez(test_file, x=x_test_1000, y=y_test_1000)

print(f'Saved training pool: {train_file}')
print(f'Saved test set: {test_file}')

Saved training pool: mnist_100_clients_60_per_class_1k_test\complete_60000_samples.npz
Saved test set: mnist_100_clients_60_per_class_1k_test\test_1000_samples.npz


In [7]:
# Split the 60k training pool into 100 balanced clients
samples_per_client = CLIENT_PER_CLASS * NUM_CLASSES
print(f'Splitting into {NUM_CLIENTS} clients with {samples_per_client} samples each...')

for client_id in range(NUM_CLIENTS):
    client_x, client_y = [], []

    for class_label in range(NUM_CLASSES):
        class_start = class_label * TRAIN_PER_CLASS
        start = class_start + client_id * CLIENT_PER_CLASS
        end = start + CLIENT_PER_CLASS

        client_x.append(x_selected[start:end])
        client_y.append(y_selected[start:end])

    client_x = np.concatenate(client_x, axis=0)
    client_y = np.concatenate(client_y, axis=0)
    client_x, client_y = shuffle(client_x, client_y, random_state=client_id)

    client_file = os.path.join(OUTPUT_DIR, f'client_{client_id + 1}.npz')
    np.savez(client_file, x=client_x, y=client_y)

    if (client_id + 1) % 10 == 0:
        print(f'Saved {client_id + 1}/{NUM_CLIENTS} clients')

print('All client files saved successfully.')

Splitting into 100 clients with 600 samples each...
Saved 10/100 clients
Saved 20/100 clients
Saved 30/100 clients
Saved 40/100 clients
Saved 50/100 clients
Saved 60/100 clients
Saved 70/100 clients
Saved 80/100 clients
Saved 90/100 clients
Saved 100/100 clients
All client files saved successfully.


In [8]:
# Quick sanity checks
client_sample = np.load(os.path.join(OUTPUT_DIR, 'client_1.npz'))

print('--- Client 1 ---')
print(f"x shape: {client_sample['x'].shape}, y shape: {client_sample['y'].shape}")
u, c = np.unique(client_sample['y'], return_counts=True)
for digit, cnt in zip(u, c):
    print(f'Digit {digit}: {cnt}')

print('--- Test 1000 ---')
u_test, c_test = np.unique(y_test_1000, return_counts=True)
for digit, cnt in zip(u_test, c_test):
    print(f'Digit {digit}: {cnt}')

--- Client 1 ---
x shape: (600, 28, 28), y shape: (600,)
Digit 0: 60
Digit 1: 60
Digit 2: 60
Digit 3: 60
Digit 4: 60
Digit 5: 60
Digit 6: 60
Digit 7: 60
Digit 8: 60
Digit 9: 60
--- Test 1000 ---
Digit 0: 100
Digit 1: 100
Digit 2: 100
Digit 3: 100
Digit 4: 100
Digit 5: 100
Digit 6: 100
Digit 7: 100
Digit 8: 100
Digit 9: 100


In [9]:
print('\n' + '=' * 60)
print('SUMMARY')
print('=' * 60)
print(f'Training pool file: {OUTPUT_DIR}/complete_60000_samples.npz')
print(f'Test set file: {OUTPUT_DIR}/test_1000_samples.npz')
print(f'Client files: {OUTPUT_DIR}/client_1.npz ... client_{NUM_CLIENTS}.npz')
print(f'Train size: {len(y_selected)}')
print(f'Test size: {len(y_test_1000)}')
print('Train/test overlap check: PASSED (0 overlapping source indices)')
print('=' * 60)


SUMMARY
Training pool file: mnist_100_clients_60_per_class_1k_test/complete_60000_samples.npz
Test set file: mnist_100_clients_60_per_class_1k_test/test_1000_samples.npz
Client files: mnist_100_clients_60_per_class_1k_test/client_1.npz ... client_100.npz
Train size: 60000
Test size: 1000
Train/test overlap check: PASSED (0 overlapping source indices)
